In [ ]:
import pandas as pd

df   = pd.read_csv('honeypot_ethics_dataset.csv', low_memory=False)
sci  = df[df['source_type'] == 'Scientific'].copy()
nons = df[df['source_type'] == 'Non-Scientific'].copy()

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total entries      : {len(df)}")
print(f"Scientific papers  : {len(sci)}")
print(f"Non-Scientific tools: {len(nons)}")

print("\n--- Domain Distribution ---")
print(df['domain'].value_counts().to_string())

print("\n--- Domain by Source Type ---")
print(df.groupby(['domain', 'source_type']).size().unstack(fill_value=0).to_string())

print("\n--- Interaction Level ---")
print(df['interaction_level'].value_counts().to_string())

print("\n--- IRB Mentioned ---")
print(df.groupby(['irb_mentioned', 'source_type']).size().unstack(fill_value=0).to_string())

print("\n--- Consent Model ---")
print(df.groupby(['consent_model', 'source_type']).size().unstack(fill_value=0).to_string())

print("\n--- Data Sensitivity ---")
print(df.groupby(['data_sensitivity', 'source_type']).size().unstack(fill_value=0).to_string())

print("\n--- Data Minimization ---")
print(df.groupby(['data_minimization', 'source_type']).size().unstack(fill_value=0).to_string())

print("\n--- Deception Framing ---")
print(df.groupby(['deception_framing', 'source_type']).size().unstack(fill_value=0).to_string())

print("\n--- Documentation Quality (Non-Scientific only) ---")
print(nons['doc_level'].value_counts().to_string())

print("\n--- Platform (Non-Scientific only) ---")
print(nons['platform'].value_counts().to_string())

print("\n--- GitHub Stars (Non-Scientific only) ---")
stars = pd.to_numeric(nons['github_stars'], errors='coerce').dropna()
print(f"Count  : {len(stars)}")
print(f"Median : {stars.median():.0f}")
print(f"Mean   : {stars.mean():.0f}")
print(f"Max    : {stars.max():.0f}")
print(f"Min    : {stars.min():.0f}")

print("\n--- Top 15 Most-Starred Tools ---")
nons2 = nons.copy()
nons2['github_stars'] = pd.to_numeric(nons2['github_stars'], errors='coerce')
top15 = nons2.nlargest(15, 'github_stars')[['identifier', 'github_stars', 'domain']]
print(top15.to_string(index=False))

print("\n--- Ethics Governance Rates (%) ---")
dims = {
    'IRB Mentioned'     : lambda r: r['irb_mentioned'] == 'Yes',
    'Consent Model'     : lambda r: str(r['consent_model']).strip() in ('Explicit', 'Implicit'),
    'Data Minimization' : lambda r: str(r['data_minimization']).strip() == 'Yes',
    'Retention Policy'  : lambda r: str(r['data_retention_policy']).strip() not in
                                     ('Not documented', 'Not discussed', '', 'nan'),
    'Legal/Jurisdiction': lambda r: str(r['jurisdiction_legal']).strip() not in
                                     ('Not discussed', 'Not documented', '', 'nan'),
    'Deception Ethics'  : lambda r: str(r['deception_framing']).strip() == 'Ethical discussion',
}
print(f"{'Dimension':<22} {'Scientific':>12} {'Non-Scientific':>16} {'All':>8}")
print("-" * 62)
for dim, fn in dims.items():
    sr = sci.apply(fn, axis=1).mean() * 100
    nr = nons.apply(fn, axis=1).mean() * 100
    ar = df.apply(fn, axis=1).mean() * 100
    print(f"{dim:<22} {sr:>11.1f}% {nr:>15.1f}% {ar:>7.1f}%")

print("\n--- Summary Statistics ---")
for src in ['Scientific', 'Non-Scientific', 'All']:
    s = df if src == 'All' else df[df['source_type'] == src]
    n = len(s)
    print(f"\n{src} (n={n}):")
    print(f"  IRB Yes             : {(s['irb_mentioned']=='Yes').sum()} ({(s['irb_mentioned']=='Yes').mean()*100:.1f}%)")
    print(f"  Consent Expl/Impl   : {s['consent_model'].isin(['Explicit','Implicit']).sum()} ({s['consent_model'].isin(['Explicit','Implicit']).mean()*100:.1f}%)")
    print(f"  Data Min Yes        : {(s['data_minimization']=='Yes').sum()} ({(s['data_minimization']=='Yes').mean()*100:.1f}%)")
    print(f"  Deception Ethics    : {(s['deception_framing']=='Ethical discussion').sum()} ({(s['deception_framing']=='Ethical discussion').mean()*100:.1f}%)")
    print(f"  High Sensitivity    : {(s['data_sensitivity']=='High').sum()} ({(s['data_sensitivity']=='High').mean()*100:.1f}%)")
    print(f"  High Interaction    : {(s['interaction_level']=='High').sum()} ({(s['interaction_level']=='High').mean()*100:.1f}%)")

DATASET OVERVIEW
Total entries      : 239
Scientific papers  : 38
Non-Scientific tools: 201

--- Domain Distribution ---
domain
IT                              190
ICS                              19
Other                            11
CPS                               8
IoT                               6
Industrial IoT (IIoT)             1
Space                             1
UAV                               1
        \nCPS / Water Plants      1
IT                                1

--- Domain by Source Type ---
source_type                   Non-Scientific  Scientific
domain                                                  
        \nCPS / Water Plants               0           1
CPS                                        3           5
ICS                                        5          14
IT                                       182           8
IT                                         0           1
Industrial IoT (IIoT)                      0           1
IoT                      